# EDA — ExpresSo NB Coffee Chain Demand Forecasting
**Task**: Forecast units_sold per (store_id, category, date) at horizons 1d / 3d / 7d  
**Window**: 2024-11-01 → 2024-12-31 · 20 stores × 7 categories × 61 days × 3 horizons = 25,620 rows  
**Metric**: MAE (lower is better)

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from pathlib import Path

plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})
sns.set_theme(style='whitegrid', palette='muted')

BASE = Path('.')
TRAIN = BASE / 'train'
TEST  = BASE / 'test'

print('Loading tables...')
TXN   = pd.read_csv(TRAIN / 'TRANSACTION.csv')
ORD   = pd.read_csv(TRAIN / 'ORDER.csv', parse_dates=['date'])
INV   = pd.read_csv(TRAIN / 'INVENTORY.csv', parse_dates=['date'])
PROD  = pd.read_csv(TRAIN / 'PRODUCT.csv')
STORE = pd.read_csv(TRAIN / 'STORE.csv')
DATE  = pd.read_csv(TRAIN / 'DATE_DIM.csv', parse_dates=['date'])
PROMO = pd.read_csv(TRAIN / 'PROMOTION.csv', parse_dates=['start_date','end_date'])
EVT   = pd.read_csv(TRAIN / 'LOCAL_EVENT.csv', parse_dates=['date'])
CUST  = pd.read_csv(TRAIN / 'CUSTOMER.csv')

SUB   = pd.read_csv(BASE / 'sample_submission_with_id.csv')

print('Done.')

## 1. Table overview

In [ ]:
tables = {
    'TRANSACTION': TXN, 'ORDER': ORD, 'INVENTORY': INV,
    'PRODUCT': PROD, 'STORE': STORE, 'DATE_DIM': DATE,
    'PROMOTION': PROMO, 'LOCAL_EVENT': EVT, 'CUSTOMER': CUST,
}

rows = []
for name, df in tables.items():
    null_pct = df.isnull().mean().mul(100).round(1)
    null_cols = null_pct[null_pct > 0].to_dict()
    rows.append({'table': name, 'rows': len(df), 'cols': df.shape[1],
                 'null_cols': null_cols})

overview = pd.DataFrame(rows)
print(overview.to_string(index=False))

In [ ]:
# Date range of ORDER table (proxy for transaction history)
print('ORDER date range:', ORD['date'].min(), '→', ORD['date'].max())
print('INV  date range:', INV['date'].min(), '→', INV['date'].max())
print('Stores in ORDER:', ORD['store_id'].nunique())
print('Products in TXN:', TXN['product_id'].nunique())
print('Categories:', PROD['category'].value_counts().to_dict())

## 2. Build the canonical target

In [ ]:
# Join TXN → ORDER (get store_id, date) → PRODUCT (get category)
target = (
    TXN
    .merge(ORD[['order_id','store_id','date']], on='order_id')
    .merge(PROD[['product_id','category']], on='product_id')
    .groupby(['store_id','category','date'])['units_sold']
    .sum()
    .reset_index()
)

print('Target shape:', target.shape)
print(target.dtypes)
print()
print(target.describe())

In [ ]:
# Fill zero-sales days: build full grid
all_dates = pd.date_range(target['date'].min(), target['date'].max(), freq='D')
stores    = target['store_id'].unique()
cats      = target['category'].unique()

full_idx  = pd.MultiIndex.from_product([stores, cats, all_dates],
                                        names=['store_id','category','date'])
full_grid = pd.DataFrame(index=full_idx).reset_index()
target_full = full_grid.merge(target, on=['store_id','category','date'], how='left')
target_full['units_sold'] = target_full['units_sold'].fillna(0)

zero_rate = (target_full['units_sold'] == 0).mean()
print(f'Zero-sales rate in full grid: {zero_rate:.1%}')
print(f'Full grid shape: {target_full.shape}')

## 3. Target distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

nonzero = target_full[target_full['units_sold'] > 0]['units_sold']

axes[0].hist(target_full['units_sold'], bins=80, color='steelblue', edgecolor='none')
axes[0].set_title('Distribution of units_sold (all rows incl. zeros)')
axes[0].set_xlabel('units_sold')

axes[1].hist(nonzero, bins=80, color='darkorange', edgecolor='none')
axes[1].set_title('Distribution of units_sold (non-zero only)')
axes[1].set_xlabel('units_sold')

axes[2].hist(np.log1p(nonzero), bins=80, color='seagreen', edgecolor='none')
axes[2].set_title('log1p(units_sold) — non-zero')
axes[2].set_xlabel('log1p(units_sold)')

plt.tight_layout()
plt.savefig('eda_01_target_dist.png', bbox_inches='tight')
plt.show()

print('\nTarget stats:')
print(target_full['units_sold'].describe().round(2))
print(f'Skewness: {target_full["units_sold"].skew():.2f}')

## 4. Overall time series

In [ ]:
daily_total = target_full.groupby('date')['units_sold'].sum().reset_index()

fig, ax = plt.subplots(figsize=(16, 4))
ax.plot(daily_total['date'], daily_total['units_sold'], lw=0.8, color='steelblue')
ax.plot(daily_total['date'],
        daily_total['units_sold'].rolling(7, center=True).mean(),
        lw=2, color='firebrick', label='7-day MA')

# Mark forecast window
ax.axvspan(pd.Timestamp('2024-11-01'), pd.Timestamp('2024-12-31'),
           alpha=0.12, color='gold', label='Forecast window')
ax.axvline(pd.Timestamp('2024-11-01'), color='orange', lw=1.5, ls='--')

ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax.xaxis.set_major_locator(mdates.MonthLocator())
plt.xticks(rotation=45)
ax.set_title('Total units sold across all stores & categories')
ax.set_ylabel('Total units_sold')
ax.legend()
plt.tight_layout()
plt.savefig('eda_02_overall_timeseries.png', bbox_inches='tight')
plt.show()

## 5. Category-level analysis

In [ ]:
cat_daily = target_full.groupby(['date','category'])['units_sold'].sum().reset_index()

fig, axes = plt.subplots(4, 2, figsize=(16, 14), sharex=True)
axes = axes.flatten()

categories = sorted(target_full['category'].unique())
colors = sns.color_palette('tab10', len(categories))

for i, (cat, col) in enumerate(zip(categories, colors)):
    df_cat = cat_daily[cat_daily['category'] == cat]
    axes[i].plot(df_cat['date'], df_cat['units_sold'], lw=0.7, color=col, alpha=0.7)
    axes[i].plot(df_cat['date'],
                 df_cat['units_sold'].rolling(7, center=True).mean(),
                 lw=1.8, color=col)
    axes[i].axvspan(pd.Timestamp('2024-11-01'), pd.Timestamp('2024-12-31'),
                    alpha=0.1, color='gold')
    axes[i].set_title(cat, fontsize=10)
    axes[i].xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))

# Hide unused subplot
for j in range(len(categories), len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Daily units sold by category (7-day MA in bold)', fontsize=13)
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('eda_03_category_timeseries.png', bbox_inches='tight')
plt.show()

# Category share
cat_total = target_full.groupby('category')['units_sold'].sum().sort_values(ascending=False)
print('\nCategory total units_sold:')
print((cat_total / cat_total.sum()).mul(100).round(1).to_string())

## 6. Store-level analysis

In [ ]:
store_total = target_full.groupby('store_id')['units_sold'].sum().sort_values(ascending=False)
store_daily_mean = target_full.groupby('store_id')['units_sold'].mean().sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

store_total.plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='none')
axes[0].set_title('Total units sold per store')
axes[0].set_xlabel('store_id')
axes[0].tick_params(axis='x', rotation=0)

store_daily_mean.plot(kind='bar', ax=axes[1], color='darkorange', edgecolor='none')
axes[1].set_title('Mean daily units sold per store')
axes[1].set_xlabel('store_id')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.savefig('eda_04_store_total.png', bbox_inches='tight')
plt.show()

# Heatmap: store × category mean sales
pivot = target_full.groupby(['store_id','category'])['units_sold'].mean().unstack()
fig, ax = plt.subplots(figsize=(12, 7))
sns.heatmap(pivot, annot=True, fmt='.1f', cmap='YlOrRd', ax=ax, linewidths=0.3)
ax.set_title('Mean daily units sold — store × category')
plt.tight_layout()
plt.savefig('eda_05_store_cat_heatmap.png', bbox_inches='tight')
plt.show()

In [ ]:
# Store attributes
print(STORE.to_string(index=False))

In [ ]:
# Merge store attributes and check correlation with sales volume
store_sales = target_full.groupby('store_id')['units_sold'].mean().reset_index()
store_sales = store_sales.merge(STORE, on='store_id')

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, col in zip(axes, ['seating_capacity','staff_count','neighborhood_type']):
    if store_sales[col].dtype == object:
        sns.barplot(data=store_sales, x=col, y='units_sold', ax=ax, palette='muted')
        ax.tick_params(axis='x', rotation=30)
    else:
        ax.scatter(store_sales[col], store_sales['units_sold'], color='steelblue')
        ax.set_xlabel(col)
        ax.set_ylabel('mean units_sold')
    ax.set_title(f'Sales vs {col}')

plt.tight_layout()
plt.savefig('eda_06_store_features.png', bbox_inches='tight')
plt.show()

## 7. Day-of-week & calendar effects

In [ ]:
# Merge DATE_DIM onto target_full
tgt = target_full.merge(DATE, on='date', how='left')

fig, axes = plt.subplots(2, 3, figsize=(16, 8))

# DoW
dow_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
dow_mean = tgt.groupby('day_of_week')['units_sold'].mean().reindex(dow_order)
axes[0,0].bar(range(7), dow_mean.values, color='steelblue')
axes[0,0].set_xticks(range(7))
axes[0,0].set_xticklabels([d[:3] for d in dow_order], rotation=30)
axes[0,0].set_title('Mean units_sold by Day-of-Week')

# Weekend vs weekday
sns.boxplot(data=tgt, x='is_weekend', y='units_sold', ax=axes[0,1], palette='muted')
axes[0,1].set_title('Weekend vs Weekday')
axes[0,1].set_ylim(0, tgt['units_sold'].quantile(0.99))

# Holiday effect
sns.boxplot(data=tgt, x='is_holiday', y='units_sold', ax=axes[0,2], palette='muted')
axes[0,2].set_title('Holiday vs Non-holiday')
axes[0,2].set_ylim(0, tgt['units_sold'].quantile(0.99))

# Payday effect
sns.boxplot(data=tgt, x='is_payday', y='units_sold', ax=axes[1,0], palette='muted')
axes[1,0].set_title('Payday effect')
axes[1,0].set_ylim(0, tgt['units_sold'].quantile(0.99))

# School break
sns.boxplot(data=tgt, x='is_school_break', y='units_sold', ax=axes[1,1], palette='muted')
axes[1,1].set_title('School break effect')
axes[1,1].set_ylim(0, tgt['units_sold'].quantile(0.99))

# Rainy season
sns.boxplot(data=tgt, x='is_rainy_season', y='units_sold', ax=axes[1,2], palette='muted')
axes[1,2].set_title('Rainy season effect')
axes[1,2].set_ylim(0, tgt['units_sold'].quantile(0.99))

plt.suptitle('Calendar effects on demand', fontsize=13)
plt.tight_layout()
plt.savefig('eda_07_calendar_effects.png', bbox_inches='tight')
plt.show()

# Print summary stats
for col in ['is_weekend','is_holiday','is_payday','is_school_break','is_rainy_season']:
    means = tgt.groupby(col)['units_sold'].mean().round(2)
    print(f'{col}: {means.to_dict()}')

In [ ]:
# Month pattern
month_mean = tgt.groupby('month')['units_sold'].mean()
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(month_mean.index, month_mean.values, color='steelblue')
ax.set_xlabel('Month')
ax.set_ylabel('Mean units_sold')
ax.set_title('Mean units_sold by Month')
ax.set_xticks(range(1, 13))
ax.set_xticklabels(['Jan','Feb','Mar','Apr','May','Jun',
                    'Jul','Aug','Sep','Oct','Nov','Dec'])
plt.tight_layout()
plt.savefig('eda_08_monthly_pattern.png', bbox_inches='tight')
plt.show()

## 8. Promotion effects

In [ ]:
print('PROMO columns:', PROMO.columns.tolist())
print(PROMO.head(3).to_string())
print('\nPromo types:', PROMO['promo_type'].value_counts().to_dict())

In [ ]:
# Expand PROMO to daily rows for store-level active promo flag
promo_rows = []
for _, row in PROMO.iterrows():
    dates = pd.date_range(row['start_date'], row['end_date'], freq='D')
    for d in dates:
        promo_rows.append({'store_id': row['store_id'], 'date': d,
                           'product_id': row['product_id'],
                           'promo_type': row['promo_type'],
                           'discount_pct': row['discount_pct']})

promo_long = pd.DataFrame(promo_rows)

# Merge product → category
promo_long = promo_long.merge(PROD[['product_id','category']], on='product_id', how='left')

# Flag: any promo active for (store, category, date)
promo_flag = (
    promo_long.groupby(['store_id','category','date'])
    .agg(has_promo=('discount_pct','count'), avg_discount=('discount_pct','mean'))
    .reset_index()
)

tgt2 = target_full.merge(promo_flag, on=['store_id','category','date'], how='left')
tgt2['has_promo'] = tgt2['has_promo'].gt(0)

promo_effect = tgt2.groupby('has_promo')['units_sold'].agg(['mean','median','std'])
print('Promotion effect on units_sold:')
print(promo_effect.round(2))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.boxplot(data=tgt2, x='has_promo', y='units_sold', ax=axes[0], palette='muted')
axes[0].set_ylim(0, tgt2['units_sold'].quantile(0.99))
axes[0].set_title('Promo vs No-promo — units_sold distribution')

promo_by_type = (promo_long.merge(
    target_full, on=['store_id','category','date'], how='left')
    .groupby('promo_type')['units_sold'].mean().sort_values(ascending=False))
promo_by_type.plot(kind='bar', ax=axes[1], color='darkorange', edgecolor='none')
axes[1].set_title('Mean units_sold by promo_type (promo days only)')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig('eda_09_promo_effects.png', bbox_inches='tight')
plt.show()

## 9. Local event effects

In [ ]:
print('Event types:', EVT['event_type'].value_counts().to_dict())

# Flag event days per store
evt_flag = EVT[['store_id','date']].copy()
evt_flag['has_event'] = True

tgt3 = target_full.merge(evt_flag, on=['store_id','date'], how='left')
tgt3['has_event'] = tgt3['has_event'].fillna(False)

evt_effect = tgt3.groupby('has_event')['units_sold'].agg(['mean','median','count'])
print('\nEvent effect on units_sold:')
print(evt_effect.round(2))

fig, ax = plt.subplots(figsize=(6, 4))
sns.boxplot(data=tgt3, x='has_event', y='units_sold', ax=ax, palette='muted')
ax.set_ylim(0, tgt3['units_sold'].quantile(0.99))
ax.set_title('Local event effect on units_sold')
plt.tight_layout()
plt.savefig('eda_10_event_effects.png', bbox_inches='tight')
plt.show()

## 10. Stockout analysis

In [ ]:
# INVENTORY.is_stockout: collapse to (store_id, date) — any product stockout
inv_stockout = (
    INV.groupby(['store_id','date'])['is_stockout']
    .any()
    .reset_index()
    .rename(columns={'is_stockout': 'any_stockout'})
)

print('Train stockout rate (store-day level):',
      inv_stockout['any_stockout'].mean().round(4))

# Category-level stockout (need product_id → category)
inv_cat = INV.merge(PROD[['product_id','category']], on='product_id')
inv_cat_stockout = (
    inv_cat.groupby(['store_id','category','date'])['is_stockout']
    .any()
    .reset_index()
    .rename(columns={'is_stockout': 'has_stockout'})
)

print('Train stockout rate (store-cat-day level):',
      inv_cat_stockout['has_stockout'].mean().round(4))

stockout_by_cat = inv_cat_stockout.groupby('category')['has_stockout'].mean().sort_values(ascending=False)
print('\nStockout rate by category:')
print(stockout_by_cat.mul(100).round(1).to_string())

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

stockout_by_cat.mul(100).plot(kind='bar', ax=axes[0], color='tomato', edgecolor='none')
axes[0].set_title('Stockout rate (%) by category')
axes[0].set_ylabel('%')
axes[0].tick_params(axis='x', rotation=30)

# Stockout effect on recorded units_sold
tgt4 = target_full.merge(inv_cat_stockout, on=['store_id','category','date'], how='left')
tgt4['has_stockout'] = tgt4['has_stockout'].fillna(False)

sns.boxplot(data=tgt4, x='has_stockout', y='units_sold', ax=axes[1], palette='muted')
axes[1].set_ylim(0, tgt4['units_sold'].quantile(0.99))
axes[1].set_title('Stockout effect on recorded units_sold')

plt.tight_layout()
plt.savefig('eda_11_stockout.png', bbox_inches='tight')
plt.show()

stk = tgt4.groupby('has_stockout')['units_sold'].mean()
print('\nMean units_sold with/without stockout:')
print(stk.round(2))

## 11. Order hour distribution (demand timing)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Orders by hour
ORD['hour'].value_counts().sort_index().plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='none')
axes[0].set_title('Orders by hour of day')
axes[0].set_xlabel('Hour')
axes[0].tick_params(axis='x', rotation=0)

# Payment method mix
ORD['payment_method'].value_counts().plot(kind='bar', ax=axes[1], color='darkorange', edgecolor='none')
axes[1].set_title('Payment method distribution')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig('eda_12_order_patterns.png', bbox_inches='tight')
plt.show()

# Walk-in rate
walkin_rate = ORD['customer_id'].isna().mean()
print(f'Walk-in (no customer_id) rate: {walkin_rate:.1%}')

## 12. Lag autocorrelation — signal for lag features

In [ ]:
from pandas.plotting import autocorrelation_plot

# Pick a representative (store, category) with dense history
ex = target_full.groupby(['store_id','category']).filter(lambda g: len(g) > 300)
grp = ex.groupby(['store_id','category'])['units_sold'].count().idxmax()
series = (target_full
          .query('store_id == @grp[0] and category == @grp[1]')
          .sort_values('date')
          .set_index('date')['units_sold'])

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# ACF manually up to 30 lags
lags = range(1, 31)
acf_vals = [series.autocorr(lag=l) for l in lags]
axes[0].bar(lags, acf_vals, color='steelblue')
axes[0].axhline(0, color='black', lw=0.8)
axes[0].axhline(1.96/np.sqrt(len(series)), color='red', ls='--', lw=0.8)
axes[0].axhline(-1.96/np.sqrt(len(series)), color='red', ls='--', lw=0.8)
axes[0].set_title(f'ACF — store {grp[0]}, category {grp[1]}')
axes[0].set_xlabel('Lag (days)')

axes[1].plot(series.index, series.values, lw=0.8, color='steelblue')
axes[1].set_title('Sample time series')
axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.xticks(rotation=30)

plt.tight_layout()
plt.savefig('eda_13_autocorrelation.png', bbox_inches='tight')
plt.show()

print(f'Peak ACF at lag 7: {series.autocorr(7):.3f}')
print(f'Peak ACF at lag 1: {series.autocorr(1):.3f}')
print(f'Peak ACF at lag 14: {series.autocorr(14):.3f}')

## 13. Submission format check

In [ ]:
print('Sample submission shape:', SUB.shape)
print(SUB.head(10).to_string())

# Parse the id column
id_parts = SUB['id'].str.rsplit('_', n=2, expand=True)
id_parts.columns = ['store_cat', 'forecast_date', 'horizon']

print('\nHorizons:', id_parts['horizon'].value_counts().to_dict())
print('Unique forecast dates:', id_parts['forecast_date'].nunique())
print('Date range:', id_parts['forecast_date'].min(), '→', id_parts['forecast_date'].max())

## 14. Key findings summary

In [ ]:
print("""
=== EDA KEY FINDINGS ===

TARGET:
  - units_sold is right-skewed; log1p transform may help tree models less but check
  - Zero-sales rate in full (store×cat×day) grid needs computing above
  - ~6.6% stockout rate in train; these truncate recorded demand → down-weight

CALENDAR SIGNALS:
  - Weekend, holiday, payday, school break all shift demand — include as features
  - Clear monthly seasonality; December (forecast period) may differ from Nov
  - Day-of-week effect is strong → encode as cyclical or one-hot

STORE SIGNALS:
  - Sales vary significantly across stores; store_id fixed effect is critical
  - Neighborhood type (university, tourist, hospital, gas_station) correlates with volume
  - Seating capacity and staff_count useful as numeric features

CATEGORY SIGNALS:
  - Coffee dominates; Bakery, Non-coffee categories have different patterns
  - Each category has its own seasonality → model per-category or use cat as feature

PROMOTION:
  - Promotions boost demand; discount_pct and promo_type are useful features
  - Test set includes planned promotions for Nov–Dec 2024

LOCAL EVENTS:
  - Events have a measurable (small) uplift; include has_event flag per store-day

LAG FEATURES:
  - Strong autocorrelation at lag 7 (weekly cycle) → lag_7, rolling_7_mean are key
  - lag_1 also significant but unavailable for 3d/7d horizons at prediction time
  - For h=7d forecast: can use lag_7, lag_14, rolling stats with min window ≥7
  - For h=1d forecast: can use lag_1, lag_7 etc.

FEATURE ENGINEERING PRIORITY:
  1. Lag features: lag_1, lag_7, lag_14, lag_28
  2. Rolling stats: rolling_7_mean, rolling_7_std, rolling_14_mean, rolling_28_mean
  3. Date features: day_of_week, month, is_weekend, is_holiday, is_payday, is_school_break
  4. Store features: neighborhood_type, seating_capacity, staff_count, has_drive_through
  5. Promo features: has_promo, avg_discount_pct, promo_type dummies
  6. Event flag: has_local_event
  7. Stockout from prior days: yesterday_stockout
  8. Identity: store_id, category (target encode or embed)
""")